# Qwen Training Agent — Google Colab

This notebook connects to your Qwen Training Engine server and uses Colab's GPU for LoRA fine-tuning.

**Flow:** Server generates training data (Gemini) → sends to this agent via WebSocket → agent trains LoRA on Colab GPU → uploads adapter back to server.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Install Dependencies

In [ ]:
!pip install -q websockets httpx pynvml psutil \
    torch transformers peft bitsandbytes datasets accelerate

## 3. Clone the Repo

In [ ]:
!rm -rf /content/qwen-training
# Replace with your repo URL:
!git clone https://github.com/YOUR_USER/qwen-training.git /content/qwen-training

## 4. Configure the Agent

**Change `SERVER_URL` to your server's public IP or ngrok URL.**

In [ ]:
import json, os

# === CHANGE THIS ===
SERVER_URL = "ws://YOUR_SERVER_IP:8000/agents/connect"
AGENT_NAME = "colab-gpu"
# ===================

config = {
    "server_url": SERVER_URL,
    "agent_name": AGENT_NAME,
    "api_key": "",
    "heartbeat_interval_seconds": 10,
    "model_cache_dir": "/content/models",
    "log_dir": "/content/logs",
}

os.makedirs("/content/models", exist_ok=True)
os.makedirs("/content/logs", exist_ok=True)

with open("/content/qwen-training/agent/config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Config written:")
print(json.dumps(config, indent=2))

## 5. Start the Agent

This cell runs indefinitely — the agent will:
1. Connect to your server via WebSocket
2. Wait for training jobs
3. Download dataset, train LoRA, upload adapter
4. Go back to waiting

**Press Stop to disconnect.**

In [ ]:
import subprocess, sys, os

os.chdir("/content/qwen-training/agent")

process = subprocess.Popen(
    [sys.executable, "main.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end="")
except KeyboardInterrupt:
    process.terminate()
    print("\n--- Agent stopped ---")